# flux-portrait — Image Generation (Colab)

Generates photos of you using your trained LoRA (`models/lora/flux_portrait_v1.safetensors`) + Flux.1 Dev, via `diffusers`.

**Before running:**
1. Runtime -> Change runtime type -> GPU (A100 or L4 both work; this notebook auto-detects VRAM and enables CPU offload on smaller GPUs).
2. Have your trained LoRA file ready to upload (`models/lora/flux_portrait_v1.safetensors` from this repo).
3. Have a HuggingFace token ready with access to `black-forest-labs/FLUX.1-dev` (same requirement as training).

**How this notebook is meant to be used:** run cells 1-6 once per session (setup: GPU check, Drive, installs, HF login, load model + LoRA). Then repeatedly edit and re-run the **generation cell** with different prompts -- that's the normal iteration loop, not a one-shot top-to-bottom run. Zip-and-download whenever you want your images off Colab.

**Rough timing**: roughly 10-20 sec/image on an A100, noticeably slower on an L4 (CPU offload trades speed for fitting in 24GB VRAM). First run also pays a one-time cost downloading Flux.1 Dev (a few GB, separate from and smaller than the raw files training needed, since `diffusers` uses its own format).

Starting prompts here mirror `configs/prompt_templates.yaml` in the repo — that file is the readable reference copy; edit prompts directly in this notebook to actually iterate.

In [ ]:
# Cell 1: GPU check + VRAM detection
import subprocess

result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
if result.returncode != 0 or not result.stdout.strip():
    raise RuntimeError(
        "No GPU detected. Go to Runtime -> Change runtime type -> select a GPU, then re-run."
    )

# Only the first GPU line is used; a multi-GPU runtime would otherwise break
# the 2-value unpack below.
first_gpu_line = result.stdout.strip().splitlines()[0]
gpu_name, mem_str = first_gpu_line.split(", ")
gpu_mem_mb = int(mem_str.strip().split(" ")[0])
LOW_VRAM = gpu_mem_mb < 32000  # True for a 24GB L4, False for a 40GB A100
print(f"Detected GPU: {gpu_name} ({gpu_mem_mb} MiB) -> LOW_VRAM={LOW_VRAM}")

In [ ]:
# Cell 2: Mount Google Drive (optional -- caches the multi-GB Flux.1 Dev
# download across sessions if it works, and lets you skip re-uploading the
# LoRA if you've saved it there before). Not a hard requirement: Colab's
# Drive auth handshake is known to be flaky, so this falls back to
# session-local storage on failure rather than blocking everything else.
import os

from google.colab import drive

DRIVE_ROOT = "/content/drive/MyDrive/flux-portrait"
try:
    drive.mount("/content/drive")
    DRIVE_MOUNTED = True
    HF_CACHE_DIR = f"{DRIVE_ROOT}/hf_cache"
    os.makedirs(HF_CACHE_DIR, exist_ok=True)
except Exception as e:
    print(f"Drive mount failed ({e}); continuing without it.")
    DRIVE_MOUNTED = False
    HF_CACHE_DIR = None  # diffusers/huggingface_hub default cache

print(f"DRIVE_MOUNTED={DRIVE_MOUNTED}, HF_CACHE_DIR={HF_CACHE_DIR}")

In [ ]:
# Cell 3: Install/upgrade inference dependencies
import subprocess

subprocess.run(
    [
        "pip", "install", "-q", "--upgrade",
        "diffusers", "transformers", "accelerate", "peft", "sentencepiece",
    ],
    check=True,
)
print("Dependencies installed.")

In [ ]:
# Cell 4: HuggingFace login (needed for the gated FLUX.1-dev repo)
import getpass

from huggingface_hub import login

hf_token = getpass.getpass("Enter your HuggingFace token (needs FLUX.1-dev access): ")
login(token=hf_token)

In [ ]:
# Cell 5: Load the Flux.1 Dev pipeline
import torch
from diffusers import FluxPipeline
from huggingface_hub.utils import HfHubHTTPError

MODEL_ID = "black-forest-labs/FLUX.1-dev"

try:
    pipe = FluxPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        cache_dir=HF_CACHE_DIR,
    )
except HfHubHTTPError as e:
    if e.response is not None and e.response.status_code == 403:
        raise RuntimeError(
            f"Access denied for {MODEL_ID}. Log into https://huggingface.co/{MODEL_ID} "
            "with the SAME account whose token you entered above, and accept its "
            "license on that page. Then re-run this cell."
        ) from e
    raise

if LOW_VRAM:
    # Trades speed for fitting the transformer + T5-XXL + CLIP-L + VAE in a
    # ~24GB card by keeping components on CPU until each is actually needed.
    pipe.enable_model_cpu_offload()
else:
    pipe.to("cuda")

print("Flux.1 Dev pipeline loaded.")

In [ ]:
# Cell 6: Upload and load your trained LoRA. Safe to re-run mid-session
# (e.g. to swap in a different .safetensors) -- unloads any previously
# loaded LoRA first so effects don't stack.
import os
import shutil

from google.colab import files

LORA_PATH = "/content/flux_portrait_v1.safetensors"

drive_lora_dir = f"{DRIVE_ROOT}/lora"
drive_lora_files = (
    [f for f in os.listdir(drive_lora_dir) if f.endswith(".safetensors")]
    if DRIVE_MOUNTED and os.path.isdir(drive_lora_dir)
    else []
)

if drive_lora_files:
    if len(drive_lora_files) > 1:
        print(f"Multiple LoRA files found on Drive, using the most recently modified: {drive_lora_files}")
    src = os.path.join(
        drive_lora_dir,
        max(drive_lora_files, key=lambda f: os.path.getmtime(os.path.join(drive_lora_dir, f))),
    )
    shutil.copy(src, LORA_PATH)
    print(f"Using LoRA found on Drive: {src}")
else:
    print("Upload your trained LoRA .safetensors file (models/lora/flux_portrait_v1.safetensors)")
    uploaded = files.upload()
    uploaded_name = next(iter(uploaded.keys()))
    shutil.move(uploaded_name, LORA_PATH)

# diffusers auto-detects and converts the Kohya/sd-scripts LoRA format
# (networks.lora_flux) this was trained with -- no manual conversion needed.
pipe.unload_lora_weights()
pipe.load_lora_weights(LORA_PATH)
print(f"LoRA loaded from {LORA_PATH}")

In [ ]:
# Cell 7: GENERATE -- edit the variables below and re-run this cell as many
# times as you want with different prompts. Each run adds new images to
# /content/output/{USE_CASE}/ rather than overwriting previous ones.
import os

import torch

PROMPT = "a photo of ohwx man sitting on a boat deck at golden hour, relaxed smile, navy casual shirt, candid"
# Matches configs/prompt_templates.yaml's negative_prompt; only actually
# applied if TRUE_CFG_SCALE > 1.0 below.
NEGATIVE_PROMPT = (
    "cartoon, illustration, painting, drawing, anime, render, deformed, ugly, "
    "blurry, low quality, watermark, text, logo, bad anatomy, extra fingers, "
    "mutated hands, poorly drawn face, bad proportions, gross proportions"
)
USE_CASE = "dating_casual"  # subfolder name under /content/output/
NUM_IMAGES = 4
SEED = None  # set an int for reproducible results, or leave None for random each run

NUM_INFERENCE_STEPS = 28
GUIDANCE_SCALE = 3.5
# Flux.1 Dev is guidance-distilled: NEGATIVE_PROMPT is silently ignored
# unless true_cfg_scale > 1.0, which then roughly doubles compute per image
# (both a conditional and unconditional pass, like classic CFG). Off by
# default to keep generation fast/cheap; raise it (e.g. 3.5) to actually
# use NEGATIVE_PROMPT.
TRUE_CFG_SCALE = 1.0
RESOLUTION = 1024

out_dir = f"/content/output/{USE_CASE}"
os.makedirs(out_dir, exist_ok=True)
# Next index = max existing + 1, not a count, so a gap left by a deleted
# image (e.g. you removed a bad 0001.png) can't cause a new image to
# silently overwrite a still-present higher-numbered file.
existing_indices = [
    int(os.path.splitext(f)[0])
    for f in os.listdir(out_dir)
    if f.endswith(".png") and os.path.splitext(f)[0].isdigit()
]
next_index = max(existing_indices, default=-1) + 1

for i in range(NUM_IMAGES):
    generator = torch.Generator().manual_seed(SEED + i) if SEED is not None else None
    kwargs = dict(
        prompt=PROMPT,
        height=RESOLUTION,
        width=RESOLUTION,
        num_inference_steps=NUM_INFERENCE_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        generator=generator,
    )
    if NEGATIVE_PROMPT and TRUE_CFG_SCALE > 1.0:
        kwargs["negative_prompt"] = NEGATIVE_PROMPT
        kwargs["true_cfg_scale"] = TRUE_CFG_SCALE

    image = pipe(**kwargs).images[0]
    out_path = os.path.join(out_dir, f"{next_index + i:04d}.png")
    image.save(out_path)
    print(f"Saved {out_path}")

print(f"Done: {NUM_IMAGES} image(s) added to {out_dir}")

In [ ]:
# Cell 8: Zip everything generated so far and download
import os
import shutil

from google.colab import files

OUTPUT_DIR = "/content/output"
if not os.path.isdir(OUTPUT_DIR) or not any(
    f.endswith(".png") for _, _, fs in os.walk(OUTPUT_DIR) for f in fs
):
    raise FileNotFoundError(
        f"No generated images found under {OUTPUT_DIR} -- run the generation "
        "cell above at least once first."
    )

zip_base = "/content/output"
zip_path = f"{zip_base}.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)
shutil.make_archive(zip_base, "zip", OUTPUT_DIR)

total_images = sum(1 for _, _, fs in os.walk(OUTPUT_DIR) for f in fs if f.endswith(".png"))
print(f"Zipped {total_images} image(s) from {OUTPUT_DIR} into {zip_path}")
files.download(zip_path)